## 잘 작동함 (반자동)

In [4]:
from selenium import webdriver 
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import pyperclip
import time
import random 
import re # 숫자 추출을 위한 정규표현식 모듈

print("=" *80)
print(" 개인프로젝트 네이버 블로그 자동 서로이웃 추가 프로그램 (우회 공격 + 팝업 제어 + 예외처리 완벽)")
print("=" *80)
print("\n")

# -------------------------------------------------------------
# 1. 사용자 입력 받기
# -------------------------------------------------------------
v_id = input('🔑 네이버 로그인 ID를 입력하세요: ')
v_passwd = input('🔑 네이버 로그인 비밀번호를 입력하세요: ')
target_count = int(input('🎯 몇 명에게 서로이웃을 신청할까요? (숫자만 입력): '))

message_text = "서로이웃해요~블로그 자주 방문하고 소통합시다!"
current_count = 0 

print("\n🚀 서로이웃 추가 자동화를 시작합니다. 브라우저가 열리면 잠시 지켜봐주세요!")

options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--disable-blink-features=AutomationControlled")

s = Service("C:/py_temp/chromedriver/chromedriver.exe")
driver = webdriver.Chrome(service=s, options=options) 

base_url = 'https://section.blog.naver.com/'
driver.get(base_url)
time.sleep(random.uniform(2, 4))
driver.maximize_window()

wait = WebDriverWait(driver, 10)
actions = ActionChains(driver)

try:
    # -------------------------------------------------------------
    # 2. 우회 로그인 처리
    # -------------------------------------------------------------
    print(">> 다이렉트 접근 및 우회 로그인 시도...")
    driver.get(f"https://blog.naver.com/{v_id}?Redirect=Write")
    time.sleep(3)

    id_element = driver.find_element(By.NAME, 'id')
    id_element.click()
    pyperclip.copy(v_id) 
    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
    time.sleep(1)

    pw_element = driver.find_element(By.NAME, 'pw')
    pw_element.click()
    pyperclip.copy(v_passwd) 
    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
    time.sleep(1)

    driver.find_element(By.ID, 'log.login').click()  
    
    # -------------------------------------------------------------
    # 💡 캡차(영수증) / 2단계 인증 대기
    # -------------------------------------------------------------
    print(">> 🚨 로그인 버튼 클릭 완료! 인증창(캡차/2단계) 발생 여부를 확인합니다...")
    print(">> 만약 캡차가 떴다면 브라우저에서 직접 마우스로 해제해주세요 (최대 5분 대기)")
    
    WebDriverWait(driver, 300).until(
        lambda d: "nid.naver.com" not in d.current_url
    )
    
    print(">> ✅ 로그인이 최종 승인되었습니다! 자동화 작업을 다시 재개합니다.")
    time.sleep(3)  
    
    # -------------------------------------------------------------
    # 3. 네이버 메인 -> 블로그 홈 -> 주제별 보기(세계여행) 이동
    # -------------------------------------------------------------
    driver.get("https://www.naver.com/")
    time.sleep(2)
    
    driver.find_element(By.XPATH, "//a[contains(@class, 'MyView-module__item_link') and .//span[text()='블로그']]").click()
    time.sleep(2)
    
    driver.find_element(By.XPATH, "//a[contains(@class, 'MyView-module__link_service') and contains(text(), '블로그')]").click()
    time.sleep(random.uniform(3, 5))
    
    driver.switch_to.window(driver.window_handles[-1])
    main_blog_window = driver.current_window_handle 
    
    wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), '주제별 보기')]"))).click()
    time.sleep(2)
    
    wait.until(EC.element_to_be_clickable((By.XPATH, "//a[.//span[text()='예술']]"))).click()
    time.sleep(3)
    
    print(">> [예술] 카테고리 진입 완료. 탐색을 시작합니다.")

    # -------------------------------------------------------------
    # 4. 반복문: 게시글 리스트 돌면서 서로이웃 신청
    # -------------------------------------------------------------
    while current_count < target_count:
        post_elements = driver.find_elements(By.CSS_SELECTOR, "a.desc_inner")
        post_urls = [elem.get_attribute("href") for elem in post_elements]
        
        for url in post_urls:
            if current_count >= target_count:
                break
            
            # 메인 창 핸들을 루프마다 명확히 재확인
            main_blog_window = driver.current_window_handle
            
            driver.execute_script(f"window.open('{url}', '_blank');")
            time.sleep(1) # 창 열릴 시간 확보
            
            try:
                # 💡 [핵심 해결 2] 인덱스가 아닌 창의 고유 ID(Handle)로 절대 추적
                post_windows = [w for w in driver.window_handles if w != main_blog_window]
                if not post_windows:
                    continue
                    
                post_window = post_windows[0]
                driver.switch_to.window(post_window)
                time.sleep(random.uniform(2, 4))
                
                # 프레임 전환
                driver.switch_to.frame("mainFrame")
                
                try:
                    add_buddy_btn = driver.find_element(By.CSS_SELECTOR, "a.btn_buddy")
                except:
                    print("   [-] 이웃추가 버튼이 없습니다. 패스합니다.")
                    continue # finally 구문으로 이동하여 창 닫기 수행

                add_buddy_btn.click()
                time.sleep(2)
                
                # 팝업 창 추적
                popup_windows = [w for w in driver.window_handles if w not in [main_blog_window, post_window]]
                if not popup_windows:
                    print("   [-] 팝업 창이 뜨지 않았습니다. 패스합니다.")
                    continue
                    
                popup_window = popup_windows[0]
                driver.switch_to.window(popup_window)
                
                # 상황별 패스
                if len(driver.find_elements(By.XPATH, "//*[contains(text(), '님과 현재 서로이웃입니다')]")) > 0:
                    print("   [-] 이미 서로이웃입니다. 패스합니다.")
                    continue
                
                if len(driver.find_elements(By.XPATH, "//*[contains(text(), '서로이웃 신청을 받지 않는 이웃입니다')]")) > 0:
                    print("   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.")
                    continue

                try:
                    both_buddy_label = wait.until(EC.element_to_be_clickable((By.XPATH, "//label[@for='each_buddy_add']")))
                    both_buddy_label.click()
                except:
                    continue
                
                time.sleep(1)
                driver.find_element(By.CSS_SELECTOR, "a._buddyAddNext").click()
                time.sleep(1.5)
                
                try:
                    textarea = driver.find_element(By.ID, "message")
                    textarea.clear()
                    
                    for char in message_text:
                        textarea.send_keys(char)
                        time.sleep(random.uniform(0.01, 0.05))
                    
                    time.sleep(1)
                    driver.find_element(By.CSS_SELECTOR, "a._addBothBuddy").click()
                    time.sleep(1.5)
                    driver.find_element(By.CSS_SELECTOR, "a.button_close").click()
                    
                    current_count += 1
                    print(f"   [+] 서로이웃 신청 완료! (현재 진행: {current_count}/{target_count})")
                    
                except:
                    print("   [-] 메시지 창을 찾을 수 없거나 에러 발생. 패스합니다.")
                
            except Exception as e:
                print("   [!] 포스팅 처리 중 에러 발생, 다음 글로 넘어갑니다.")
                
            finally:
                # 💡 [핵심 해결 2] 어떤 창이 꼬이더라도 무조건 '메인 창'만 남기고 싹 다 닫는 불도저 로직
                for handle in driver.window_handles:
                    if handle != main_blog_window:
                        try:
                            driver.switch_to.window(handle)
                            driver.close()
                        except:
                            pass # 이미 닫힌 창이면 무시
                # 안전하게 다시 메인 창으로 복귀
                driver.switch_to.window(main_blog_window)
                time.sleep(random.uniform(1.5, 3))
                
        # 10개(한 페이지)를 다 돌았는데 아직 목표치에 도달하지 못했다면 페이징 처리
        if current_count < target_count:
            print(">> 한 페이지를 모두 탐색했습니다. 다음 페이지로 이동합니다.")
            
            try:
                # 💡 [핵심 해결 1] 상단 메뉴의 "주제별 보기"와 혼동하지 않도록 하단 페이징 영역으로 한정
                pagination_area = driver.find_element(By.CSS_SELECTOR, "div.pagination")
                current_page_elem = pagination_area.find_element(By.CSS_SELECTOR, "a[aria-current='page']")
                
                # 텍스트에서 숫자만 완벽하게 추출 (예: " 1 " -> 1)
                current_page_text = current_page_elem.text
                current_page_num = int(re.sub(r'[^0-9]', '', current_page_text))
                next_page_num = current_page_num + 1
                
                driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", pagination_area)
                time.sleep(1.5)
                
                try:
                    # 다음 숫자 페이지 버튼 찾아서 클릭
                    next_page_btn = pagination_area.find_element(By.CSS_SELECTOR, f"a[aria-label='{next_page_num}페이지']")
                    try:
                        next_page_btn.click()
                    except:
                        driver.execute_script("arguments[0].click();", next_page_btn)
                except:
                    # 10페이지 단위가 넘어가서 숫자가 안보일 경우 '다음' 화살표 그룹 버튼 클릭
                    next_group_btn = pagination_area.find_element(By.CSS_SELECTOR, "a.button_next")
                    try:
                        next_group_btn.click()
                    except:
                        driver.execute_script("arguments[0].click();", next_group_btn)
                
                time.sleep(4) # 페이지 로딩 대기
                
            except Exception as page_e:
                print(">> 페이징 처리 중 마지막 페이지에 도달했거나 에러가 발생했습니다. 프로그램을 종료합니다.", page_e)
                break

    print("\n🎉 목표한 서로이웃 신청 횟수를 모두 채웠습니다! 프로그램을 종료합니다.")
    time.sleep(5)

except Exception as e:
    print("\n[치명적 에러 발생] 프로그램 실행 도중 문제가 발생했습니다:", e)

finally:
    driver.quit()

 개인프로젝트 네이버 블로그 자동 서로이웃 추가 프로그램 (우회 공격 + 팝업 제어 + 예외처리 완벽)



🚀 서로이웃 추가 자동화를 시작합니다. 브라우저가 열리면 잠시 지켜봐주세요!
>> 다이렉트 접근 및 우회 로그인 시도...
>> 🚨 로그인 버튼 클릭 완료! 인증창(캡차/2단계) 발생 여부를 확인합니다...
>> 만약 캡차가 떴다면 브라우저에서 직접 마우스로 해제해주세요 (최대 5분 대기)
>> ✅ 로그인이 최종 승인되었습니다! 자동화 작업을 다시 재개합니다.
>> [예술] 카테고리 진입 완료. 탐색을 시작합니다.
   [!] 포스팅 처리 중 에러 발생, 다음 글로 넘어갑니다.
   [+] 서로이웃 신청 완료! (현재 진행: 1/50)
   [-] 메시지 창을 찾을 수 없거나 에러 발생. 패스합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 2/50)
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 3/50)
   [+] 서로이웃 신청 완료! (현재 진행: 4/50)
   [-] 메시지 창을 찾을 수 없거나 에러 발생. 패스합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 5/50)
>> 한 페이지를 모두 탐색했습니다. 다음 페이지로 이동합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 6/50)
   [-] 메시지 창을 찾을 수 없거나 에러 발생. 패스합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 7/50)
   [+] 서로이웃 신청 완료! (현재 진행: 8/50)
   [+] 서로이웃 신청 완료! (현재 진행: 9/50)
   [-] 메시지 창을 찾을 수 없거나 에러 발생. 패스합니다.
   [-] 메시지 창을 찾을 수 없거나 에러 발생. 패스합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 10/50)
   [-] 메시지 창을 찾을 수 없거나 에러 발생. 패스합니다.
   [-]

KeyboardInterrupt: 

In [1]:
from selenium import webdriver 
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import pyperclip
import time
import random 
import re # 숫자 추출을 위한 정규표현식 모듈

print("=" *80)
print(" 개인프로젝트 네이버 블로그 자동 서로이웃 추가 프로그램 (우회 공격 + 팝업 제어 + 예외처리 완벽)")
print("=" *80)
print("\n")

# -------------------------------------------------------------
# 1. 사용자 입력 받기
# -------------------------------------------------------------
v_id = input('🔑 네이버 로그인 ID를 입력하세요: ')
v_passwd = input('🔑 네이버 로그인 비밀번호를 입력하세요: ')
target_count = int(input('🎯 몇 명에게 서로이웃을 신청할까요? (숫자만 입력): '))

message_text = "서로이웃해요~블로그 자주 방문하고 소통합시다!"
current_count = 0 

print("\n🚀 서로이웃 추가 자동화를 시작합니다. 브라우저가 열리면 잠시 지켜봐주세요!")

options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--disable-blink-features=AutomationControlled")

s = Service("C:/py_temp/chromedriver/chromedriver.exe")
driver = webdriver.Chrome(service=s, options=options) 

base_url = 'https://section.blog.naver.com/'
driver.get(base_url)
time.sleep(random.uniform(2, 4))
driver.maximize_window()

wait = WebDriverWait(driver, 10)
actions = ActionChains(driver)

try:
    # -------------------------------------------------------------
    # 2. 우회 로그인 처리
    # -------------------------------------------------------------
    print(">> 다이렉트 접근 및 우회 로그인 시도...")
    driver.get(f"https://blog.naver.com/{v_id}?Redirect=Write")
    time.sleep(3)

    id_element = driver.find_element(By.NAME, 'id')
    id_element.click()
    pyperclip.copy(v_id) 
    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
    time.sleep(1)

    pw_element = driver.find_element(By.NAME, 'pw')
    pw_element.click()
    pyperclip.copy(v_passwd) 
    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
    time.sleep(1)

    driver.find_element(By.ID, 'log.login').click()  
    
    # -------------------------------------------------------------
    # 💡 캡차(영수증) / 2단계 인증 대기
    # -------------------------------------------------------------
    print(">> 🚨 로그인 버튼 클릭 완료! 인증창(캡차/2단계) 발생 여부를 확인합니다...")
    print(">> 만약 캡차가 떴다면 브라우저에서 직접 마우스로 해제해주세요 (최대 5분 대기)")
    
    WebDriverWait(driver, 300).until(
        lambda d: "nid.naver.com" not in d.current_url
    )
    
    print(">> ✅ 로그인이 최종 승인되었습니다! 자동화 작업을 다시 재개합니다.")
    time.sleep(3)  
    
    # -------------------------------------------------------------
    # 3. 네이버 메인 -> 블로그 홈 -> 주제별 보기(예술 -> 영화) 이동
    # -------------------------------------------------------------
    driver.get("https://www.naver.com/")
    time.sleep(2)
    
    driver.find_element(By.XPATH, "//a[contains(@class, 'MyView-module__item_link') and .//span[text()='블로그']]").click()
    time.sleep(2)
    
    driver.find_element(By.XPATH, "//a[contains(@class, 'MyView-module__link_service') and contains(text(), '블로그')]").click()
    time.sleep(random.uniform(3, 5))
    
    driver.switch_to.window(driver.window_handles[-1])
    main_blog_window = driver.current_window_handle 
    
    wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), '주제별 보기')]"))).click()
    time.sleep(2)
    
    wait.until(EC.element_to_be_clickable((By.XPATH, "//a[.//span[text()='예술']]"))).click()
    time.sleep(2)
    print(">> [예술] 카테고리 진입 완료.")
    
    # [추가된 로직]: 예술 카테고리 진입 후 '영화' 탭 클릭
    print(">> [영화] 서브 카테고리로 이동합니다...")
    movie_tab = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(@class, 'item') and .//span[text()='영화']]")))
    try:
        movie_tab.click()
    except:
        # AngularJS 렌더링 충돌 방지용 자바스크립트 강제 클릭
        driver.execute_script("arguments[0].click();", movie_tab)
    
    time.sleep(3) # 영화 카테고리의 게시글 목록이 로드될 때까지 충분히 대기
    print(">> [영화] 게시글 탐색을 시작합니다.")

    # -------------------------------------------------------------
    # 4. 반복문: 게시글 리스트 돌면서 서로이웃 신청
    # -------------------------------------------------------------
    while current_count < target_count:
        post_elements = driver.find_elements(By.CSS_SELECTOR, "a.desc_inner")
        post_urls = [elem.get_attribute("href") for elem in post_elements]
        
        for url in post_urls:
            if current_count >= target_count:
                break
            
            # 메인 창 핸들을 루프마다 명확히 재확인
            main_blog_window = driver.current_window_handle
            
            driver.execute_script(f"window.open('{url}', '_blank');")
            time.sleep(1) # 창 열릴 시간 확보
            
            try:
                # 💡 [핵심 해결 2] 인덱스가 아닌 창의 고유 ID(Handle)로 절대 추적
                post_windows = [w for w in driver.window_handles if w != main_blog_window]
                if not post_windows:
                    continue
                    
                post_window = post_windows[0]
                driver.switch_to.window(post_window)
                time.sleep(random.uniform(2, 4))
                
                # 프레임 전환
                driver.switch_to.frame("mainFrame")
                
                try:
                    add_buddy_btn = driver.find_element(By.CSS_SELECTOR, "a.btn_buddy")
                except:
                    print("   [-] 이웃추가 버튼이 없습니다. 패스합니다.")
                    continue # finally 구문으로 이동하여 창 닫기 수행

                add_buddy_btn.click()
                time.sleep(2)
                
                # 팝업 창 추적
                popup_windows = [w for w in driver.window_handles if w not in [main_blog_window, post_window]]
                if not popup_windows:
                    print("   [-] 팝업 창이 뜨지 않았습니다. 패스합니다.")
                    continue
                    
                popup_window = popup_windows[0]
                driver.switch_to.window(popup_window)
                
                # 상황별 패스
                if len(driver.find_elements(By.XPATH, "//*[contains(text(), '님과 현재 서로이웃입니다')]")) > 0:
                    print("   [-] 이미 서로이웃입니다. 패스합니다.")
                    continue
                
                if len(driver.find_elements(By.XPATH, "//*[contains(text(), '서로이웃 신청을 받지 않는 이웃입니다')]")) > 0:
                    print("   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.")
                    continue

                try:
                    both_buddy_label = wait.until(EC.element_to_be_clickable((By.XPATH, "//label[@for='each_buddy_add']")))
                    both_buddy_label.click()
                except:
                    continue
                
                time.sleep(1)
                driver.find_element(By.CSS_SELECTOR, "a._buddyAddNext").click()
                time.sleep(1.5)
                
                try:
                    # 1. 텍스트 에어리어 로딩 대기 및 초기화
                    textarea = wait.until(EC.presence_of_element_located((By.ID, "message")))
                    textarea.clear()
                    time.sleep(0.5)
                    
                    # [핵심 해결 로직]
                    # send_keys()로 한 글자씩 타이핑하는 방식은 최신 웹의 동적 렌더링과 충돌하여 
                    # 중간에 튕기는 현상(StaleElement 예외)을 유발합니다.
                    # 사람처럼 입력하는 척하는 것보다, 클립보드로 한 번에 '붙여넣기' 하는 것이 
                    # 네이버의 JS 이벤트 꼬임을 방지하고 봇 탐지를 우회하는 데 훨씬 안정적입니다.
                    textarea.click()
                    pyperclip.copy(message_text)
                    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
                    time.sleep(1) # 붙여넣기 인식 대기
                    
                    # 2. '확인(서로이웃 신청)' 버튼 명시적 대기 후 클릭
                    submit_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a._addBothBuddy")))
                    try:
                        submit_btn.click()
                    except:
                        # 일반적인 click()이 먹히지 않을 경우 JS로 강제 클릭 (예외 상황 완벽 제어)
                        driver.execute_script("arguments[0].click();", submit_btn)
                    
                    time.sleep(2) # 서버 전송 대기
                    
                    # 3. '닫기' 버튼 명시적 대기 후 클릭
                    close_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.button_close")))
                    try:
                        close_btn.click()
                    except:
                        driver.execute_script("arguments[0].click();", close_btn)
                    
                    current_count += 1
                    print(f"   [+] 서로이웃 신청 완료! (현재 진행: {current_count}/{target_count})")
                    
                except Exception as inner_e:
                    # 어떤 에러 때문에 튕겼는지 이유를 출력하도록 수정 (디버깅의 기본)
                    print(f"   [-] 메시지 창 에러 발생. 사유: {inner_e}")
                
            except Exception as e:
                print("   [!] 포스팅 처리 중 에러 발생, 다음 글로 넘어갑니다.")
                
            finally:
                # 💡 [핵심 해결 2] 어떤 창이 꼬이더라도 무조건 '메인 창'만 남기고 싹 다 닫는 불도저 로직
                for handle in driver.window_handles:
                    if handle != main_blog_window:
                        try:
                            driver.switch_to.window(handle)
                            driver.close()
                        except:
                            pass # 이미 닫힌 창이면 무시
                # 안전하게 다시 메인 창으로 복귀
                driver.switch_to.window(main_blog_window)
                time.sleep(random.uniform(1.5, 3))
                
        # 10개(한 페이지)를 다 돌았는데 아직 목표치에 도달하지 못했다면 페이징 처리
        if current_count < target_count:
            print(">> 한 페이지를 모두 탐색했습니다. 다음 페이지로 이동합니다.")
            
            try:
                # 💡 [핵심 해결 1] 상단 메뉴의 "주제별 보기"와 혼동하지 않도록 하단 페이징 영역으로 한정
                pagination_area = driver.find_element(By.CSS_SELECTOR, "div.pagination")
                current_page_elem = pagination_area.find_element(By.CSS_SELECTOR, "a[aria-current='page']")
                
                # 텍스트에서 숫자만 완벽하게 추출 (예: " 1 " -> 1)
                current_page_text = current_page_elem.text
                current_page_num = int(re.sub(r'[^0-9]', '', current_page_text))
                next_page_num = current_page_num + 1
                
                driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", pagination_area)
                time.sleep(1.5)
                
                try:
                    # 다음 숫자 페이지 버튼 찾아서 클릭
                    next_page_btn = pagination_area.find_element(By.CSS_SELECTOR, f"a[aria-label='{next_page_num}페이지']")
                    try:
                        next_page_btn.click()
                    except:
                        driver.execute_script("arguments[0].click();", next_page_btn)
                except:
                    # 10페이지 단위가 넘어가서 숫자가 안보일 경우 '다음' 화살표 그룹 버튼 클릭
                    next_group_btn = pagination_area.find_element(By.CSS_SELECTOR, "a.button_next")
                    try:
                        next_group_btn.click()
                    except:
                        driver.execute_script("arguments[0].click();", next_group_btn)
                
                time.sleep(4) # 페이지 로딩 대기
                
            except Exception as page_e:
                print(">> 페이징 처리 중 마지막 페이지에 도달했거나 에러가 발생했습니다. 프로그램을 종료합니다.", page_e)
                break

    print("\n🎉 목표한 서로이웃 신청 횟수를 모두 채웠습니다! 프로그램을 종료합니다.")
    time.sleep(5)

except Exception as e:
    print("\n[치명적 에러 발생] 프로그램 실행 도중 문제가 발생했습니다:", e)

finally:
    driver.quit()

 개인프로젝트 네이버 블로그 자동 서로이웃 추가 프로그램 (우회 공격 + 팝업 제어 + 예외처리 완벽)



🚀 서로이웃 추가 자동화를 시작합니다. 브라우저가 열리면 잠시 지켜봐주세요!
>> 다이렉트 접근 및 우회 로그인 시도...
>> 🚨 로그인 버튼 클릭 완료! 인증창(캡차/2단계) 발생 여부를 확인합니다...
>> 만약 캡차가 떴다면 브라우저에서 직접 마우스로 해제해주세요 (최대 5분 대기)
>> ✅ 로그인이 최종 승인되었습니다! 자동화 작업을 다시 재개합니다.
>> [예술] 카테고리 진입 완료.
>> [영화] 서브 카테고리로 이동합니다...
>> [영화] 게시글 탐색을 시작합니다.
   [!] 포스팅 처리 중 에러 발생, 다음 글로 넘어갑니다.
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 1/50)
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 2/50)
   [+] 서로이웃 신청 완료! (현재 진행: 3/50)
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 4/50)
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.
>> 한 페이지를 모두 탐색했습니다. 다음 페이지로 이동합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 5/50)
   [+] 서로이웃 신청 완료! (현재 진행: 6/50)
   [+] 서로이웃 신청 완료! (현재 진행: 7/50)
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.
   [+] 서로이웃 신청 완료! (현재 진행: 8/50)
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다.
   [-] 서로이웃 신청을 받지 않는 블로거입니다. 패스합니다

KeyboardInterrupt: 